# Why KTO?

You already understand DPO as:

> Given two answers, tell me which one is better.

**Example:**

- **Prompt:** "How do I learn Python?"
- **A:** detailed, useful explanation 👍
- **B:** vague, useless answer 👎
- **Preference:** A > B

That's pairwise preference feedback.

---

## KTO changes the question

KTO says:

> We don't always need two answers. Just tell me whether this one answer is good or bad.

So our dataset can simply be:

| Prompt | Response | Label |
|--------|----------|-------|
| "How do I learn Python?" | "Start with..." | 👍 |
| "How do I learn Python?" | "I don't know lol" | 👎 |
| "Explain recursion" | "Recursion is..." | 👍 |
| "Explain recursion" | "idk" | 👎 |

**That's it.**

---

## Why is this useful?

Real users don't naturally give you:

> Answer A > Answer B

They more naturally give:

- 👍 I like this
- 👎 I don't like this

For example, a chatbot could collect thousands of interactions where users simply upvote/downvote responses.

**KTO is designed to learn from that kind of binary feedback.**

---

## The important connection to RLHF

Remember our earlier confusion about reward models?

**KTO still doesn't need a separate reward model.**

We have:

```
user feedback
     ↓
   👍 / 👎
     ↓
KTO objective
     ↓
update LLM
```

**NOT:**

```
LLM → Reward Model → reward → PPO
```

And unlike DPO, we don't need:

> chosen + rejected

We only need:

> response + binary label
```

---

## One thing to NOT misunderstand

**KTO is NOT simply binary classification.**

The 👍/👎 is the feedback label.

The model still produces probabilities/log-probabilities for the response, and KTO uses those probabilities to construct its training objective.

## How does KTO get a learning signal?

Here's the key mental model.

**With DPO**, we had:

```
chosen vs rejected
        ↓
compare their log-probs
        ↓
which one should move up/down?
```

**With KTO**, we only have one response + 👍/👎.

So we need another way to decide:

> How good is this response relative to what the model normally does?

**And this is where the reference policy comes back.**

---

### Step 1 — Get the policy probability

Suppose:

- **Prompt:** "Explain photosynthesis"
- **Response:** "Plants convert light energy into chemical energy..."

Our current model gives that response some probability:

$$ \pi_\theta(y|x) $$

Same idea you already learned:

> π = probability assigned by the current LLM to generating that response.

---

### Step 2 — Get the reference probability

The frozen reference model gives:

$$ \pi_{ref}(y|x) $$

This tells us:

> "How likely was this response under the original model?"

So now we can compare:

```
Current model       Reference model
      ↓                    ↓
 πθ(y|x)              πref(y|x)
      └────── compare ─────┘
```

---

### Step 3 — Why compare them?

Imagine the current model starts making a response much more likely than the original model.

That tells us:

> The policy has moved toward this response.

But KTO needs to know whether that movement is good or bad.

**That's where the user's 👍 / 👎 comes in.**

Conceptually:

```
             response
                ↓
       ┌────────┴────────┐
       ↓                 ↓
   πθ(response)     πref(response)
       └────────┬────────┘
                ↓
          compare them
                ↓
          use 👍 / 👎
                ↓
            KTO loss
```

---

### And here's the important bit

KTO also uses a **KL term** to measure how far the policy has generally drifted from the reference model.

So there are actually **two ideas**:

1. This particular response: how did $\pi_\theta$ change relative to $\pi_{\rm ref}$?
2. Overall policy: how far has $\pi_\theta$ drifted from $\pi_{\rm ref}$?

The KL part acts somewhat like an anchor:

> "Improve according to feedback, but don't go completely crazy and destroy the original model."

---

### Conceptual jump

**DPO:**

A vs B → preference

**KTO:**

```
A → 👍/👎
   +
policy vs reference
   +
KL anchor
   ↓
learning signal
```

## The KTO equation

Now we get to the actual math. Don't memorize it yet; understand what each piece is doing.

For a response $y$ to prompt $x$, KTO essentially builds a reward-like quantity:

$$ r(x,y) = \beta \left( \log \frac{\pi_\theta(y|x)} {\pi_{\mathrm{ref}}(y|x)} - z_{\mathrm{ref}} \right) $$

There are **3 important pieces**.

---

## 1. The probability ratio

$$ \log \frac{\pi_\theta(y|x)} {\pi_{\mathrm{ref}}(y|x)} $$

This asks:

> "How much more/less does my current model like this response compared with the original model?"

**If:** $\pi_\theta > \pi_{\rm ref}$

→ the current model has increased its probability.

**If:** $\pi_\theta < \pi_{\rm ref}$

→ it has decreased it.

---

## 2. The KL anchor

$z_{\mathrm{ref}}$ is related to the **KL divergence** between the current policy and reference policy.

Think of it simply as:

> "How much has my model generally moved away from the reference?"

This prevents KTO from saying:

> "👍 this one response is good → massively increase it."

Instead, the model is encouraged to improve while staying reasonably close to its original behavior.

---

## 3. The 👍 / 👎 changes the direction

This is the really important part.

**KTO has two cases:**

**👍 desirable response**
```
       ↓
increase its relative probability
```

**👎 undesirable response**
```
       ↓
decrease its relative probability
```

The loss uses a sigmoid-style objective to make that happen.

Conceptually:

$$ \text{KTO loss} = \begin{cases} -\sigma(r) & \text{👍}\\ -\sigma(-r) & \text{👎} \end{cases} $$

Don't worry about the exact implementation yet.

---

## The mental model

Just remember:

```
response
   ↓
πθ / πref
   ↓
"did current model move toward this?"
   ↓
     👍              👎
     ↓               ↓
reward movement    punish movement
     ↓               ↓
          KTO loss
```

**And this is why KTO can learn from individual 👍/👎 responses without needing a chosen-vs-rejected pair.**

## The important bit

KTO is basically doing this:

```
             Reference model
                   ↓
prompt → response → compare probabilities
                   ↓
          ┌────────┴────────┐
          ↓                 ↓
         👍                👎
          ↓                 ↓
    increase it        decrease it
```

And $z_{\rm ref}$/KL anchoring prevents the model from going absolutely feral and changing its behavior everywhere just because of a few feedback labels.

---

## DPO vs KTO

| | DPO | KTO |
|-------|-----|-----|
| **Feedback** | chosen vs rejected | 👍 / 👎 |
| **Pair required?** | ✅ | ❌ |
| **Reference model** | ✅ | ✅ |
| **Reward model** | ❌ | ❌ |
| **Core signal** | relative preference | desirability |

---

### Conceptual jump

**DPO:** "Between these two answers, which one wins?"

**KTO:** "Is this answer good or bad?"